# Compare denoising vs frozen AR teacher distillation

Runs **Llama 3.2 1B Instruct** twice, from the same pretrained checkpoint:

- **Baseline:** usual denoising loss only.
- **Mixed:** 50% denoising loss + 50% full-vocabulary teacher KL, temperature 1.

Both runs use the same seed, first 1,000 training examples, batch size 1, sequence length 256, rank-128 LoRA, and 1,000 training updates. The frozen teacher uses clean labels and causal attention, with LoRA disabled. Validation uses the original denoising objective for both models.

Select **Runtime → Change runtime type → GPU**, then **Runtime → Run all**. Accept the [Llama model's access terms](https://huggingface.co/meta-llama/Llama-3.2-1B-Instruct) and add a read token as `HF_TOKEN` in Colab Secrets, or sign in when prompted. Drive saving is enabled by default and will request access.

This notebook uses only the experimental branch of `lad-generic`. The public BYOD reproduction is unchanged.


## 1. Install

Fetch only the experiment folder and reuse Colab’s installed GPU packages.


In [ ]:
import importlib
from importlib import metadata
import os, subprocess, sys, time
from pathlib import Path
from tempfile import TemporaryDirectory

setup_started = time.monotonic()

REPOSITORY_URL = "https://github.com/RuurdKuiper/lad-generic.git"
EXPERIMENT_BRANCH = "experiment/ar-distillation"
REPOSITORY_DIR = Path("/content/lad-generic-distillation")
PROJECT_DIR = REPOSITORY_DIR / "byod-distillation"

def setup_step(label, command):
    print(f"{label}…", flush=True)
    started = time.monotonic()
    subprocess.run(command, check=True)
    print(f"{label} completed in {time.monotonic() - started:.1f}s", flush=True)

if not (REPOSITORY_DIR / ".git").exists():
    setup_step("Clone experiment", ["git", "clone", "--depth", "1", "--filter=blob:none", "--sparse",
                             "--branch", EXPERIMENT_BRANCH, REPOSITORY_URL, str(REPOSITORY_DIR)])
else:
    setup_step("Update experiment", ["git", "-C", str(REPOSITORY_DIR), "pull", "--ff-only"])
setup_step("Select experimental source", ["git", "-C", str(REPOSITORY_DIR), "sparse-checkout", "set", "byod-distillation"])
os.chdir(PROJECT_DIR)
# Keep the runtime's working GPU stack instead of downloading replacement
# Torch/CUDA wheels. Other dependencies are still resolved normally by pip.
# If a future dependency needs a different Torch, report the conflict explicitly.
gpu_constraints = []
for distribution in metadata.distributions():
    name = distribution.metadata["Name"].lower().replace("_", "-")
    if name in {"torch", "torchvision", "torchaudio", "triton"} or name.startswith("nvidia-"):
        gpu_constraints.append(f"{name}=={distribution.version}")
print("Keeping installed GPU packages:", ", ".join(sorted(gpu_constraints)) or "none", flush=True)
with TemporaryDirectory(prefix="byod-setup-") as temporary_dir:
    constraints = Path(temporary_dir) / "gpu-constraints.txt"
    constraints.write_text("\n".join(sorted(gpu_constraints)) + "\n")
    setup_step("Install training package and widgets", [
        sys.executable, "-m", "pip", "install", "--prefer-binary",
        "--upgrade-strategy", "only-if-needed", "--constraint", str(constraints),
        "-e", ".[colab]",
    ])

# PEFT can reject an old preinstalled TorchAO even when quantization is disabled.
# BYOD does not need TorchAO; only remove the known-incompatible versions.
from packaging.version import Version
try:
    torchao_version = metadata.version("torchao")
except metadata.PackageNotFoundError:
    torchao_version = None
if torchao_version is not None and Version(torchao_version) < Version("0.16.0"):
    setup_step(f"Remove incompatible optional TorchAO {torchao_version}", [
        sys.executable, "-m", "pip", "uninstall", "-y", "torchao",
    ])

# Editable installs create a .pth file that this already-running kernel may
# not have processed. Make this checkout importable immediately.
for path in (PROJECT_DIR, PROJECT_DIR / "src"):
    if str(path) not in sys.path:
        sys.path.insert(0, str(path))
importlib.invalidate_caches()
import diffusion_lm
print(f"Ready in {time.monotonic() - setup_started:.1f}s. Gradio installs only in step 6.", flush=True)


## 2. Hugging Face access


In [ ]:
import os
from huggingface_hub import login

try:
    from google.colab import userdata
    token = userdata.get("HF_TOKEN")
except Exception:
    token = None

if token:
    os.environ["HF_TOKEN"] = token
    login(token=token, add_to_git_credential=False)
else:
    login(add_to_git_credential=False)

## 3. Comparison settings

Defaults run the requested 1,000-update comparison. For a smoke test, change `UPDATES` to 2. Held-out examples and deterministic corruption are identical across runs. Validation/checkpoint selection uses the original denoising loss, rather than the mixed training objective.

The 64-example validation/test subsets make this a quick pilot. Keep the same GPU runtime for both runs. The teacher adds work per update; the plots compare both convergence per update and elapsed time.


In [ ]:
UPDATES = 1000
EVALUATION_SAMPLES = 64
VALIDATION_STEPS = 100
SAVE_TO_DRIVE = True


## 4. Run both experiments

The baseline runs first, then the mixed objective in a fresh process. Each has a live dashboard showing mixed, denoising, and distillation losses. The shared model/data download happens once. A new comparison directory is created each time this cell runs; previous results are preserved. Stopping this cell interrupts training and saves the partial adapter after completed updates.


In [ ]:
import importlib
import datetime
import uuid
from pathlib import Path
import torch

if not torch.cuda.is_available():
    raise RuntimeError("Select a GPU runtime under Runtime > Change runtime type.")
print("GPU:", torch.cuda.get_device_name(0))

from diffusion_lm.notebook import remove_unused_transformer_engine
remove_unused_transformer_engine()

if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    comparison_root = Path("/content/drive/MyDrive/BYOD-distillation-comparisons")
else:
    comparison_root = Path("/content/BYOD-distillation-comparisons")

stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M%S")
output_root = comparison_root / f"llama1b-{UPDATES}-{stamp}-{uuid.uuid4().hex[:6]}"
print("Comparison directory:", output_root)

import run_comparison as comparison_helpers
importlib.reload(comparison_helpers)
summary = comparison_helpers.run_comparison(
    output_root, updates=UPDATES, evaluation_samples=EVALUATION_SAMPLES,
    validation_steps=VALIDATION_STEPS, dashboard=True,
)


## 5. Compare the results

The plots show **the same ordinary validation denoising loss** for both runs. Lower loss is better; reaching it earlier on the time axis is evidence of faster convergence in this pilot. Training time includes validation but excludes model/data setup. One seed and small held-out subsets do not establish a general improvement.

The table distinguishes best validation loss from final validation loss. Final test loss describes the final adapter. Saved `best/` adapters were selected using ordinary validation loss. The JSON summary, PNG chart, configs, adapters, and complete logs remain in the comparison directory.


In [ ]:
import json
from IPython.display import Image, Markdown, display

# To revisit an earlier completed comparison, set output_root to its directory.
summary = json.loads((output_root / "summary.json").read_text())
rows = [
    "| Run | Final validation loss | Best validation loss | Final test loss | Training + validation minutes |",
    "| --- | ---: | ---: | ---: | ---: |",
]
for name, label in (("baseline", "Denoising only"), ("distilled", "50/50 mixed")):
    result = summary[name]
    rows.append(
        f"| {label} | {result['final_validation']['weighted_loss']:.4f} "
        f"| {result['best_validation']['weighted_loss']:.4f} "
        f"| {result['final_test']['weighted_loss']:.4f} "
        f"| {result['final_validation']['elapsed_seconds'] / 60:.2f} |"
    )
report = "\n".join(rows)
(output_root / "comparison.md").write_text(report + "\n")
display(Markdown(report))
display(Image(filename=str(output_root / "comparison.png")))
print("All results:", output_root)
